# **Setup**

In [25]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [26]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [27]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [28]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on kaggle — storage at: /kaggle/working


# **Load Data**

In [29]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [30]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [31]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

optimizer = ModelOptimizer("ItemKNN_dice")

STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + '_dice'

In [32]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "dice",
        "topK": optuna_trial.suggest_int("topK", 20, 500),
        "shrink": optuna_trial.suggest_int("shrink", 0, 1000),
        "normalize": optuna_trial.suggest_categorical("normalize", [True, False]),
        "feature_weighting": optuna_trial.suggest_categorical("feature_weighting", ["none", "TF-IDF", "BM25"]),
    }

    if params["feature_weighting"] == "BM25":
        params["BM25_k1"] = optuna_trial.suggest_float("BM25_k1", 0.5, 2.0)
        params["BM25_b"] = optuna_trial.suggest_float("BM25_b", 0.0, 1.0)
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [33]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-19 23:05:30,271] A new study created in RDB with name: ItemKNNCFRecommender_dice


  0%|          | 0/100 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1937.40 column/sec. Elapsed time 3.60 sec
  Fold 1/5 - Score: 0.20125861854840071
Similarity column 6969 (100.0%), 1951.44 column/sec. Elapsed time 3.57 sec
  Fold 2/5 - Score: 0.20206635903253492
Similarity column 6969 (100.0%), 1966.55 column/sec. Elapsed time 3.54 sec
  Fold 3/5 - Score: 0.2026571909110625
Similarity column 6969 (100.0%), 1965.13 column/sec. Elapsed time 3.55 sec
  Fold 4/5 - Score: 0.20172339896236766
Similarity column 6969 (100.0%), 1964.15 column/sec. Elapsed time 3.55 sec
  Fold 5/5 - Score: 0.2021663319810426
[I 2025-11-19 23:07:39,237] Trial 0 finished with value: 0.20197437988708167 and parameters: {'topK': 313, 'shrink': 514, 'normalize': False, 'feature_weighting': 'BM25', 'BM25_k1': 0.5120381511018068, 'BM25_b': 0.7102491082641964}. Best is trial 0 with value: 0.20197437988708167.
Similarity column 6969 (100.0%), 1953.41 column/sec. Elapsed time 3.57 sec
  Fold 1/5 - Score: 0.1920375764287409
Similarity column 6969 (100.0%)

In [34]:
optuna.visualization.plot_optimization_history(optuna_study)

In [35]:
optuna.visualization.plot_param_importances(optuna_study)

In [36]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- Best Value: 0.20997681478499564
Best Params: {'topK': 20, 'shrink': 202, 'normalize': False, 'feature_weighting': 'BM25', 'BM25_k1': 1.8418189912262135, 'BM25_b': 0.04185180282545142}